In [2]:
# Cell 1: Read the silver_sales Delta table
df_silver = spark.table("ws_atharv_sales.lh_sales.dbo.silver_sales")

print(f"Row count: {df_silver.count()}")
print(f"Columns: {df_silver.columns}")
display(df_silver.limit(5))

StatementMeta(, 422032c2-8e31-474f-9922-49a99c2f308f, 4, Finished, Available, Finished, False)

Row count: 1268
Columns: ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'revenue', 'quantity', 'status', 'revenue_usd']


SynapseWidget(Synapse.DataFrame, 834360c6-3750-481b-af3f-3a304c6ef4d0)

In [3]:
# Cell 2: Extract year and month from order_date
from pyspark.sql.functions import col, date_format, sum, count, round, avg

df_dated = df_silver.withColumn(
    'year_month',
    date_format(col('order_date'), 'yyyy-MM')
)
display(df_dated.select('order_id', 'order_date', 'year_month').limit(5))

StatementMeta(, 422032c2-8e31-474f-9922-49a99c2f308f, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1407d546-0bdd-4658-80f7-08e82d709dd3)

In [5]:
# Cell 3: Aggregate to gold layer
df_gold = df_dated.groupBy('region', 'year_month') \
    .agg(
        round(sum('revenue'), 2).alias('total_revenue'),
        count('order_id').alias('total_orders'),
        round(avg('revenue'), 2).alias('avg_order_value')
    ) \
    .orderBy('year_month', 'region')

print(f'Gold table row count: {df_gold.count()}')
display(df_gold)

StatementMeta(, 422032c2-8e31-474f-9922-49a99c2f308f, 7, Finished, Available, Finished, False)

Gold table row count: 96


SynapseWidget(Synapse.DataFrame, 4d1646cc-7273-4f83-866c-6c64af62e7a3)

In [7]:
# Cell 4: Write the gold layer to a delta table
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema","true") \
    .saveAsTable('ws_atharv_sales.lh_sales.dbo.gold_sales_summary')

StatementMeta(, 422032c2-8e31-474f-9922-49a99c2f308f, 9, Finished, Available, Finished, False)